**Navigation** : [jumeau Python](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Python.ipynb) · [jumeau Rust](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Rust.ipynb)

# Chaînes 11 — Le groupe de Lorentz et la signature un contre trois

Chapitre 50 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu), sections 54-55. Deux résultats : le **Théorème 54.1** (conditionnel) — le groupe de Lorentz comme isométries de la scission hermitienne — et le **Théorème 55.1**, premier résultat de l'arc au statut de théorème établi : la signature un contre trois vit au choix de l'unité et le transport la porte le long de chaque chaîne sans bouger.

Le patron de la série tient : **Mesuré** ce que nos rationnels exacts rejouent, **Cité** toute la chaîne dérivative du source, **Ouvert** ce qu'il ne prouve pas.

Ce carnet est le **jumeau C#** de [Chaînes 11 — version Python](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Python.ipynb) : mêmes entrées, mêmes valeurs, et une arithmétique entièrement exacte portée par `System.Numerics.BigInteger` et un rationnel exact réduit (numérateur et dénominateur entiers).

## Le cadre : l'ordre rejoint les rotations

« The rotations fixed, the next chapter joins them to the order coordinate in the Lorentz group » (p. 143). Le chapitre 49 avait dérivé les rotations des directions ; celui-ci y adjoint les boosts — les générateurs qui **mélangent** l'ordre et la direction.

Les statuts diffèrent et le carnet les garde visibles : le 54.1 est **conditionnel** (« conditional on the axiom text and on the results of the chapters named as its premises »), le 55.1 porte la mention « **Standing: theorem** » — la première de la Partie IV sur ce tronçon.

## §54 — Théorème 54.1 (conditionnel) : le groupe de Lorentz comme isométries de la scission hermitienne

« The isometry algebra of the Hermitian split at the qutrit-qubit link [...] is six-dimensional, spanned by the ad matrices of the directions, which fix the unit, and by three boosts generated from the hyperbolic generator by the rotation brackets; at every structure constant it equals the Lorentz algebra of the literature by name, with the Killing form of signature three and three, and the six lift to the qubit end as three anti-Hermitian and three Hermitian generators » (p. 144).

**Notre modèle, déclaré nôtre** : l'espace $\mathbb{R}^4$ de coordonnées $(t, x, y, z)$ muni de $\eta = \mathrm{diag}(4, -1, -1, -1)$ — **la Gram de propagation de l'éq. 81 elle-même** —, trois rotations de bloc spatial (les $\mathrm{ad}$ matrices des directions de la construction, citées), trois boosts dont le bloc $2\times2$ est le générateur littéral imprimé de l'éq. 82. Tout ce qui suit se mesure en **rationnel exact** (numérateur et dénominateur `BigInteger`) sur ce modèle ; l'égalité « by name » à l'algèbre de la construction reste citée.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Arithmetique exacte sur Q : matrices 4x4 en tableaux imbriques de rationnel exact.
// Rationnel exact reduit (numerateur / denominateur BigInteger) : le pendant C# de fractions.Fraction.
public readonly struct RationnelC03 : IEquatable<RationnelC03>
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }

    public RationnelC03(BigInteger numerateur) : this(numerateur, BigInteger.One) { }

    public RationnelC03(BigInteger numerateur, BigInteger denominateur)
    {
        if (denominateur == BigInteger.Zero)
            throw new DivideByZeroException("Denominateur nul : rationnel indefini.");
        if (denominateur < BigInteger.Zero)
        {
            numerateur = -numerateur;
            denominateur = -denominateur;
        }
        BigInteger g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(numerateur), denominateur);
        if (g == BigInteger.Zero) g = BigInteger.One;
        Numerateur = numerateur / g;
        Denominateur = denominateur / g;
    }

    public static RationnelC03 operator +(RationnelC03 a, RationnelC03 b) =>
        new RationnelC03(a.Numerateur * b.Denominateur + b.Numerateur * a.Denominateur,
                        a.Denominateur * b.Denominateur);

    public static RationnelC03 operator -(RationnelC03 a, RationnelC03 b) =>
        new RationnelC03(a.Numerateur * b.Denominateur - b.Numerateur * a.Denominateur,
                        a.Denominateur * b.Denominateur);

    public static RationnelC03 operator *(RationnelC03 a, RationnelC03 b) =>
        new RationnelC03(a.Numerateur * b.Numerateur, a.Denominateur * b.Denominateur);

    public static RationnelC03 operator *(RationnelC03 a, BigInteger k) =>
        new RationnelC03(k * a.Numerateur, a.Denominateur);

    public static RationnelC03 operator *(BigInteger k, RationnelC03 a) => a * k;

    public static RationnelC03 operator /(RationnelC03 a, RationnelC03 b) =>
        new RationnelC03(a.Numerateur * b.Denominateur, a.Denominateur * b.Numerateur);

    public static RationnelC03 operator -(RationnelC03 a) =>
        new RationnelC03(-a.Numerateur, a.Denominateur);

    public static bool operator ==(RationnelC03 a, RationnelC03 b) =>
        a.Numerateur == b.Numerateur && a.Denominateur == b.Denominateur;

    public static bool operator !=(RationnelC03 a, RationnelC03 b) => !(a == b);

    public bool Equals(RationnelC03 autre) => this == autre;

    public override bool Equals(object? obj) => obj is RationnelC03 r && Equals(r);

    public override int GetHashCode() => HashCode.Combine(Numerateur, Denominateur);

    public override string ToString() => Denominateur == BigInteger.One
        ? Numerateur.ToString(CultureInfo.InvariantCulture)
        : Numerateur.ToString(CultureInfo.InvariantCulture) + "/" + Denominateur.ToString(CultureInfo.InvariantCulture);
}

// Verdict executable : leve si la valeur calculee differe de l'attendu.
public static class VerifC03
{
    public static void Egale(bool condition, string message)
    {
        if (!condition) throw new InvalidOperationException("ECART : " + message);
    }
}

// Fabriques de rationnels : F(5, 4) vaut 5/4, F(-3, 2) vaut -3/2.
public static class FxC03
{
    public static RationnelC03 F(BigInteger n) => new RationnelC03(n);
    public static RationnelC03 F(BigInteger n, BigInteger d) => new RationnelC03(n, d);
}

// Matrices en tableaux imbriques de rationnels exacts : mul, add, sub, br, eq,
// transp, neg, zeros, E -- toute comparaison est coefficient par coefficient.
public static class MatC03
{
    public static RationnelC03[][] Zeros(int n)
    {
        var m = new RationnelC03[n][];
        for (int i = 0; i < n; i++)
        {
            m[i] = new RationnelC03[n];
            for (int j = 0; j < n; j++) m[i][j] = new RationnelC03(BigInteger.Zero);
        }
        return m;
    }

    public static RationnelC03[][] E(int n, int i, int j)
    {
        var m = Zeros(n);
        m[i][j] = new RationnelC03(BigInteger.One);
        return m;
    }

    public static RationnelC03[][] Mul(RationnelC03[][] A, RationnelC03[][] B)
    {
        int n = A.Length, q = A[0].Length, p = B[0].Length;
        var m = new RationnelC03[n][];
        for (int i = 0; i < n; i++)
        {
            m[i] = new RationnelC03[p];
            for (int j = 0; j < p; j++)
            {
                RationnelC03 s = new RationnelC03(BigInteger.Zero);
                for (int k = 0; k < q; k++) s = s + A[i][k] * B[k][j];
                m[i][j] = s;
            }
        }
        return m;
    }

    public static RationnelC03[][] Add(RationnelC03[][] A, RationnelC03[][] B)
    {
        int n = A.Length, p = A[0].Length;
        var m = new RationnelC03[n][];
        for (int i = 0; i < n; i++)
        {
            m[i] = new RationnelC03[p];
            for (int j = 0; j < p; j++) m[i][j] = A[i][j] + B[i][j];
        }
        return m;
    }

    public static RationnelC03[][] Sub(RationnelC03[][] A, RationnelC03[][] B)
    {
        int n = A.Length, p = A[0].Length;
        var m = new RationnelC03[n][];
        for (int i = 0; i < n; i++)
        {
            m[i] = new RationnelC03[p];
            for (int j = 0; j < p; j++) m[i][j] = A[i][j] - B[i][j];
        }
        return m;
    }

    public static RationnelC03[][] Br(RationnelC03[][] A, RationnelC03[][] B) => Sub(Mul(A, B), Mul(B, A));

    public static bool Eq(RationnelC03[][] A, RationnelC03[][] B)
    {
        if (A.Length != B.Length || A[0].Length != B[0].Length) return false;
        for (int i = 0; i < A.Length; i++)
            for (int j = 0; j < A[0].Length; j++)
                if (!(A[i][j] == B[i][j])) return false;
        return true;
    }

    public static RationnelC03[][] Transp(RationnelC03[][] A)
    {
        int n = A.Length, p = A[0].Length;
        var m = new RationnelC03[p][];
        for (int i = 0; i < p; i++)
        {
            m[i] = new RationnelC03[n];
            for (int j = 0; j < n; j++) m[i][j] = A[j][i];
        }
        return m;
    }

    public static RationnelC03[][] Neg(RationnelC03[][] A)
    {
        int n = A.Length, p = A[0].Length;
        var m = new RationnelC03[n][];
        for (int i = 0; i < n; i++)
        {
            m[i] = new RationnelC03[p];
            for (int j = 0; j < p; j++) m[i][j] = -A[i][j];
        }
        return m;
    }

    public static int NonNuls(RationnelC03[][] A)
    {
        int compte = 0;
        for (int i = 0; i < A.Length; i++)
            for (int j = 0; j < A[0].Length; j++)
                if (A[i][j] != new RationnelC03(BigInteger.Zero)) compte++;
        return compte;
    }

    public static bool Nulle(RationnelC03[][] A) => NonNuls(A) == 0;

    public static string Aff(RationnelC03[][] A)
    {
        var lignes = new List<string>();
        for (int i = 0; i < A.Length; i++)
        {
            var cases = new List<string>();
            for (int j = 0; j < A[0].Length; j++) cases.Add(A[i][j].ToString());
            lignes.Add(string.Join(", ", cases));
        }
        return "[" + string.Join("; ", lignes) + "]";
    }
}

// Trois boosts (plans (1/2, 2)).
public static class BoostC03
{
    public static RationnelC03[][] Boost(int i)
    {
        var K = MatC03.Zeros(4);
        K[0][i] = FxC03.F(1, 2);
        K[i][0] = FxC03.F(2);
        return K;
    }
}

// Condition d'algebre : X^T eta + eta X = 0 pour les six.
public static class ResC03
{
    public static RationnelC03[][] Residu(RationnelC03[][] X, RationnelC03[][] eta) =>
        MatC03.Add(MatC03.Mul(MatC03.Transp(X), eta), MatC03.Mul(eta, X));
}

// L'appariement : eta = diag(4, -1, -1, -1) -- la Gram de propagation de l'eq. 81.
var eta = MatC03.Zeros(4);
eta[0][0] = FxC03.F(4);
eta[1][1] = FxC03.F(-1);
eta[2][2] = FxC03.F(-1);
eta[3][3] = FxC03.F(-1);

// Trois rotations (blocs spatiaux, (Ji)_jk = -eps_ijk) et trois boosts (plans (1/2, 2)).
var J1 = MatC03.Sub(MatC03.E(4, 3, 2), MatC03.E(4, 2, 3));
var J2 = MatC03.Sub(MatC03.E(4, 1, 3), MatC03.E(4, 3, 1));
var J3 = MatC03.Sub(MatC03.E(4, 2, 1), MatC03.E(4, 1, 2));
var K1 = BoostC03.Boost(1);
var K2 = BoostC03.Boost(2);
var K3 = BoostC03.Boost(3);
var baseGen = new[] { J1, J2, J3, K1, K2, K3 };
var noms = new[] { "J1", "J2", "J3", "K1", "K2", "K3" };

Console.WriteLine("Condition d'algebre : X^T eta + eta X = 0 pour les six generateurs.");
var residus = new Dictionary<string, bool>();
int entreesNonNules = 0;
for (int k = 0; k < baseGen.Length; k++)
{
    var R = ResC03.Residu(baseGen[k], eta);
    int nn = MatC03.NonNuls(R);
    entreesNonNules += nn;
    bool nul = nn == 0;
    residus[noms[k]] = nul;
    Console.WriteLine($"  {noms[k]} : entrees non nulles du residu = {nn} (attendu 0) -> {(nul ? "CONFORME" : "ECART")}");
    VerifC03.Egale(nul, $"residu X^T eta + eta X nul pour {noms[k]} (entrees non nulles : {nn})");
}
Console.WriteLine($"Six residus nuls : {residus.Values.Count(v => v)} / 6 ; entrees non nulles cumulees : {entreesNonNules}");
VerifC03.Egale(entreesNonNules == 0, "les six residus d'isometrie doivent etre nuls");
Console.WriteLine($"Bloc 2x2 du boost K1 (eq. 82) : [0, {K1[0][1]}; {K1[1][0]}, 0]");


Condition d'algebre : X^T eta + eta X = 0 pour les six generateurs.


  J1 : entrees non nulles du residu = 0 (attendu 0) -> CONFORME


  J2 : entrees non nulles du residu = 0 (attendu 0) -> CONFORME


  J3 : entrees non nulles du residu = 0 (attendu 0) -> CONFORME


  K1 : entrees non nulles du residu = 0 (attendu 0) -> CONFORME


  K2 : entrees non nulles du residu = 0 (attendu 0) -> CONFORME


  K3 : entrees non nulles du residu = 0 (attendu 0) -> CONFORME


Six residus nuls : 6 / 6 ; entrees non nulles cumulees : 0


Bloc 2x2 du boost K1 (eq. 82) : [0, 1/2; 2, 0]


### Lecture du résultat (isométries)

**Mesuré** : les six générateurs — trois rotations qui fixent l'axe du temps (les « ad matrices of the directions » de la construction, citées) et trois boosts — satisfont la condition d'algèbre d'isométrie de $\eta$. Le bloc $2\times2$ de chaque boost, $\begin{psmallmatrix} 0 & 1/2 \\ 2 & 0 \end{psmallmatrix}$, est le générateur imprimé de l'éq. 82 : notre modèle et la construction partagent les mêmes matrices de plan.

In [2]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Les crochets de l'eq. 80 : [J1,J2] = J3, [Ji,Kj] = eps_ijk Kk, [K1,K2] = -J3.
// eps_ijk sur les indices {1,2,3}.
public static class EpsC05
{
    public static int Eps(int i, int j, int k)
    {
        if (i == j || j == k || i == k) return 0;
        if ((i == 1 && j == 2 && k == 3) || (i == 2 && j == 3 && k == 1) || (i == 3 && j == 1 && k == 2)) return 1;
        return -1;
    }
}

// Les rotations se ferment sur elles-memes : [J1,J2] = J3, [J2,J3] = J1, [J3,J1] = J2.
var libellesRot = new[] { "[J1,J2]=J3", "[J2,J3]=J1", "[J3,J1]=J2" };
var rotRot = new List<bool>
{
    MatC03.Eq(MatC03.Br(J1, J2), J3),
    MatC03.Eq(MatC03.Br(J2, J3), J1),
    MatC03.Eq(MatC03.Br(J3, J1), J2),
};
for (int k = 0; k < rotRot.Count; k++)
{
    Console.WriteLine($"  {libellesRot[k]} -> {(rotRot[k] ? "CONFORME" : "ECART")}");
    VerifC03.Egale(rotRot[k], $"crochet {libellesRot[k]}");
}
Console.WriteLine($"Crochets rotation-rotation conformes : {rotRot.Count(v => v)} / 3");

// Les neuf crochets rotation-boost : [Ji,Kj] = eps_ijk Kk.
int melangeConformes = 0;
for (int i = 1; i <= 3; i++)
    for (int j = 1; j <= 3; j++)
    {
        var attendu = MatC03.Zeros(4);
        for (int k = 1; k <= 3; k++)
        {
            int e = EpsC05.Eps(i, j, k);
            if (e != 0)
            {
                var Kk = baseGen[3 + k - 1];
                for (int r = 0; r < 4; r++)
                    for (int c = 0; c < 4; c++)
                        attendu[r][c] = attendu[r][c] + Kk[r][c] * FxC03.F(e);
            }
        }
        bool ok = MatC03.Eq(MatC03.Br(baseGen[i - 1], baseGen[3 + j - 1]), attendu);
        if (ok) melangeConformes++;
        Console.WriteLine($"  [J{i},K{j}] = eps_ijk K_k -> {(ok ? "CONFORME" : "ECART")}");
        VerifC03.Egale(ok, $"crochet [J{i},K{j}] = eps_ijk K_k");
    }
var melangeOk = melangeConformes == 9;
Console.WriteLine($"Crochets rotation-boost conformes : {melangeConformes} / 9");
VerifC03.Egale(melangeOk, "les neuf crochets [Ji,Kj] = eps_ijk K_k doivent etre conformes");

// Deux boosts donnent moins une rotation : [K1,K2] = -J3, [K2,K3] = -J1, [K3,K1] = -J2.
var libellesBoost = new[] { "[K1,K2]=-J3", "[K2,K3]=-J1", "[K3,K1]=-J2" };
var boostBoost = new List<bool>
{
    MatC03.Eq(MatC03.Br(K1, K2), MatC03.Neg(J3)),
    MatC03.Eq(MatC03.Br(K2, K3), MatC03.Neg(J1)),
    MatC03.Eq(MatC03.Br(K3, K1), MatC03.Neg(J2)),
};
for (int k = 0; k < boostBoost.Count; k++)
{
    Console.WriteLine($"  {libellesBoost[k]} -> {(boostBoost[k] ? "CONFORME" : "ECART")}");
    VerifC03.Egale(boostBoost[k], $"crochet {libellesBoost[k]}");
}
Console.WriteLine($"Crochets boost-boost conformes : {boostBoost.Count(v => v)} / 3");
VerifC03.Egale(boostBoost.Count(v => v) == 3, "les trois crochets [K1,K2]=-J3, [K2,K3]=-J1, [K3,K1]=-J2 doivent etre conformes");


  [J1,J2]=J3 -> CONFORME


  [J2,J3]=J1 -> CONFORME


  [J3,J1]=J2 -> CONFORME


Crochets rotation-rotation conformes : 3 / 3


  [J1,K1] = eps_ijk K_k -> CONFORME


  [J1,K2] = eps_ijk K_k -> CONFORME


... (8 outputs left out)

  [K1,K2]=-J3 -> CONFORME


  [K2,K3]=-J1 -> CONFORME


  [K3,K1]=-J2 -> CONFORME


Crochets boost-boost conformes : 3 / 3


### Lecture du résultat (constantes de structure)

**Mesuré** : l'éq. 80 exacte sur notre modèle — les rotations se ferment sur elles-mêmes, les crochets rotation-boost **font tourner le boost** (les neuf $[J_i, K_j] = \epsilon_{ijk} K_k$), et deux boosts donnent **moins une rotation**. « at every structure constant it equals the Lorentz algebra of the literature by name » : ici ce sont nos propres rationnels exacts qui font le compte.

In [3]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// L'identite de Jacobi sur les 216 triples de la base.
int jacobiTotal = 0;
int jacobiConformes = 0;
foreach (var X in baseGen)
    foreach (var Y in baseGen)
        foreach (var Z in baseGen)
        {
            var somme = MatC03.Add(
                MatC03.Add(MatC03.Br(MatC03.Br(X, Y), Z), MatC03.Br(MatC03.Br(Y, Z), X)),
                MatC03.Br(MatC03.Br(Z, X), Y));
            jacobiTotal++;
            if (MatC03.Nulle(somme)) jacobiConformes++;
        }
var jacobi = jacobiConformes == 216;
Console.WriteLine($"Jacobi : {jacobiConformes} / {jacobiTotal} triples d'identite nulle");
VerifC03.Egale(jacobiTotal == 216, $"216 triples de la base attendus, obtenu {jacobiTotal}");
VerifC03.Egale(jacobi, $"l'identite de Jacobi doit tenir sur les 216 triples, conformes : {jacobiConformes}");
Console.WriteLine($"Jacobi 216/216 triples -> {(jacobi ? "CONFORME" : "ECART")}");


Jacobi : 216 / 216 triples d'identite nulle


Jacobi 216/216 triples -> CONFORME


### Lecture du résultat

**Mesuré** : l'identité de Jacobi tient sur les 216 triples de la base — le crochet est une structure d'algèbre de Lie, pas seulement quinze égalités isolées.

In [4]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Forme de Killing k(X,Y) = Tr(ad_X ad_Y) dans la base (J1..J3, K1..K3).
// Les coefficients d'un crochet sont des entiers (-1, 0, 1) ; la Killing est exacte en rationnel.
public static class KillingC09
{
    // Coefficients d'une matrice C dans la base (J1..J3, K1..K3) ; null si C n'est pas dans la base.
    public static BigInteger[]? Coeffs(RationnelC03[][] C, RationnelC03[][][] baseGen)
    {
        if (MatC03.Nulle(C)) return new BigInteger[6];
        for (int k = 0; k < baseGen.Length; k++)
        {
            if (MatC03.Eq(C, baseGen[k]))
            {
                var v = new BigInteger[6];
                v[k] = BigInteger.One;
                return v;
            }
            if (MatC03.Eq(C, MatC03.Neg(baseGen[k])))
            {
                var v = new BigInteger[6];
                v[k] = BigInteger.MinusOne;
                return v;
            }
        }
        return null;
    }

    // Signature d'une forme symetrique par elimination symetrique exacte (pivots).
    public static (int Positifs, int Negatifs) SignatureSym(RationnelC03[][] S)
    {
        int n = S.Length;
        var M = new RationnelC03[n][];
        for (int i = 0; i < n; i++)
        {
            M[i] = new RationnelC03[n];
            for (int j = 0; j < n; j++) M[i][j] = S[i][j];
        }
        var signes = new List<int>();
        while (signes.Count < n)
        {
            int pr = -1;
            var best = new RationnelC03(BigInteger.Zero);
            for (int r = 0; r < M.Length; r++)
            {
                var diag = M[r][r];
                var abs = diag.Numerateur < BigInteger.Zero ? -diag : diag;
                // Les denominateurs sont positifs : la comparaison exacte se fait par produits croises.
                if (abs.Numerateur * best.Denominateur > best.Numerateur * abs.Denominateur)
                {
                    best = abs;
                    pr = r;
                }
            }
            if (pr < 0 || M[pr][pr] == new RationnelC03(BigInteger.Zero)) break;
            signes.Add(M[pr][pr].Numerateur > BigInteger.Zero ? 1 : -1);
            var p = M[pr][pr];
            var garde = new List<int>();
            for (int i = 0; i < M.Length; i++) if (i != pr) garde.Add(i);
            var suivant = new RationnelC03[garde.Count][];
            for (int ii = 0; ii < garde.Count; ii++)
            {
                suivant[ii] = new RationnelC03[garde.Count];
                for (int jj = 0; jj < garde.Count; jj++)
                {
                    var terme = M[garde[ii]][pr] * M[pr][garde[jj]] / p;
                    suivant[ii][jj] = M[garde[ii]][garde[jj]] - terme;
                }
            }
            M = suivant;
        }
        return (signes.Count(s => s == 1), signes.Count(s => s == -1));
    }
}

// Les six matrices ad_X : colonnes = coefficients de [X, e] dans la base.
var ads = new BigInteger[6][][];
bool fermeture = true;
for (int a = 0; a < 6; a++)
{
    ads[a] = new BigInteger[6][];
    for (int e = 0; e < 6; e++)
    {
        var v = KillingC09.Coeffs(MatC03.Br(baseGen[a], baseGen[e]), baseGen);
        if (v == null)
        {
            fermeture = false;
            v = new BigInteger[6];
        }
        ads[a][e] = v;
    }
}
Console.WriteLine($"Fermeture de la base (les 36 crochets sont dans la base) -> {(fermeture ? "CONFORME" : "ECART")}");
VerifC03.Egale(fermeture, "la base (J1..J3, K1..K3) doit etre fermee sous le crochet");

// kappa[a][b] = Tr(ad_a ad_b) = somme_j somme_k ads[a][j][k] * ads[b][k][j].
var kappa = new RationnelC03[6][];
for (int a = 0; a < 6; a++)
{
    kappa[a] = new RationnelC03[6];
    for (int b = 0; b < 6; b++)
    {
        BigInteger somme = BigInteger.Zero;
        for (int j = 0; j < 6; j++)
            for (int k = 0; k < 6; k++)
                somme += ads[a][j][k] * ads[b][k][j];
        kappa[a][b] = new RationnelC03(somme);
    }
}

var diagonale = new List<string>();
for (int i = 0; i < 6; i++) diagonale.Add(kappa[i][i].ToString());
Console.WriteLine($"kappa diagonale (J1..J3, K1..K3) : [{string.Join(", ", diagonale)}]");

// Valeurs attendues : -4 pour chaque rotation, +4 pour chaque boost.
var attendus = new[] { -4, -4, -4, 4, 4, 4 };
bool kappaDiagOk = true;
for (int i = 0; i < 6; i++)
{
    var attendu = new RationnelC03(attendus[i]);
    bool ok = kappa[i][i] == attendu;
    if (!ok) kappaDiagOk = false;
    Console.WriteLine($"  kappa({noms[i]},{noms[i]}) = {kappa[i][i]} (attendu {attendu}) -> {(ok ? "CONFORME" : "ECART")}");
    VerifC03.Egale(ok, $"kappa diagonale de {noms[i]} : {attendu} attendu");
}

// Tous les termes croises sont nuls.
int croisesNonNuls = 0;
for (int i = 0; i < 6; i++)
    for (int j = 0; j < 6; j++)
        if (i != j && kappa[i][j] != new RationnelC03(BigInteger.Zero)) croisesNonNuls++;
Console.WriteLine($"kappa hors-diagonale : {croisesNonNuls} entrees non nulles (attendu 0) -> {(croisesNonNuls == 0 ? "CONFORME" : "ECART")}");
VerifC03.Egale(croisesNonNuls == 0, $"les termes croises de kappa doivent etre nuls, non nuls : {croisesNonNuls}");

// Signature par elimination symetrique exacte.
var sigKappa = KillingC09.SignatureSym(kappa);
bool signatureOk = sigKappa == (3, 3);
Console.WriteLine($"Signature de kappa (pivots symetriques) : ({sigKappa.Positifs}, {sigKappa.Negatifs}) (attendu (3, 3)) -> {(signatureOk ? "CONFORME" : "ECART")}");
VerifC03.Egale(signatureOk, $"signature de kappa : (3, 3) attendu, obtenu ({sigKappa.Positifs}, {sigKappa.Negatifs})");
Console.WriteLine($"Killing mesuree : diag(-4,-4,-4,4,4,4), signature (3, 3) -> {(kappaDiagOk && croisesNonNuls == 0 && signatureOk ? "CONFORME" : "ECART")}");

Fermeture de la base (les 36 crochets sont dans la base) -> CONFORME


kappa diagonale (J1..J3, K1..K3) : [-4, -4, -4, 4, 4, 4]


  kappa(J1,J1) = -4 (attendu -4) -> CONFORME


  kappa(J2,J2) = -4 (attendu -4) -> CONFORME


  kappa(J3,J3) = -4 (attendu -4) -> CONFORME


  kappa(K1,K1) = 4 (attendu 4) -> CONFORME


  kappa(K2,K2) = 4 (attendu 4) -> CONFORME


  kappa(K3,K3) = 4 (attendu 4) -> CONFORME


kappa hors-diagonale : 0 entrees non nulles (attendu 0) -> CONFORME


Signature de kappa (pivots symetriques) : (3, 3) (attendu (3, 3)) -> CONFORME


Killing mesuree : diag(-4,-4,-4,4,4,4), signature (3, 3) -> CONFORME


### Lecture du résultat (Killing)

**Mesuré** : $\kappa(J,J) = -4$ pour chaque rotation, $\kappa(K,K) = +4$ pour chaque boost, tous les termes croisés nuls — la forme de Killing est $\mathrm{diag}(-4,-4,-4,\,4,4,4)$, signature **(3, 3)** par élimination symétrique exacte. Les sous-espaces se séparent d'eux-mêmes : les rotations portent (0, 3), les boosts (3, 0) — et la moitié (0, 3) est précisément la lecture du contre-test du source (« with the unit struck the six-dimensional algebra moves to the three-dimensional isometry algebra of the directions alone at the signature zero and three », p. 145, cité).

## Les Grams des deux liens : l'éq. 79 et l'éq. 81

À l'unité hermitienne du lien qutrit-qubit (dimension six) : « the unit squaring to the link dimension six, every direction step to minus three halves and every interference term zero » (p. 145) — l'éq. 79, $G = \tfrac32\,\mathrm{diag}(4,-1,-1,-1)$. À la chaîne série (2,2), la Gram de propagation (éq. 81) est $\mathrm{diag}(4,-1,-1,-1)$, signature $(1,3)$, le rapport $-4$.

**Le rapport est le même aux deux types de lien** : $6 \div \tfrac32 = 4 \div 1 = 4$ — l'unité pèse quatre directions, quel que soit le lien. Cité : la Gram calculée cellule par cellule aux **six types de paire**, la même scission partout.

In [5]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Eq. 79 : G = diag(6, -3/2, -3/2, -3/2) = (3/2)*diag(4,-1,-1,-1) ; eq. 81 : diag(4,-1,-1,-1).
// Signes des pivots et criteres de Sylvester (mineurs principaux) sur une diagonale.
public static class GramC12
{
    public static (int Positifs, int Negatifs) SigPivots(RationnelC03[] diag)
    {
        int pos = 0, neg = 0;
        foreach (var v in diag)
        {
            // Le denominateur d'un rationnel reduit est strictement positif.
            if (v.Numerateur > BigInteger.Zero) pos++;
            else neg++;
        }
        return (pos, neg);
    }

    public static (int Positifs, int Negatifs) SigSylvester(RationnelC03[] diag)
    {
        var prods = new List<RationnelC03>();
        var p = new RationnelC03(BigInteger.One);
        foreach (var v in diag)
        {
            p = p * v;
            prods.Add(p);
        }
        var suite = new List<int> { 1 };
        foreach (var x in prods)
            suite.Add(x.Numerateur > BigInteger.Zero ? 1 : -1);
        int negatifs = 0;
        for (int i = 0; i + 1 < suite.Count; i++) if (suite[i] != suite[i + 1]) negatifs++;
        return (diag.Length - negatifs, negatifs);
    }
}

var G79 = new[] { FxC03.F(6), FxC03.F(-3, 2), FxC03.F(-3, 2), FxC03.F(-3, 2) };
var G81 = new[] { FxC03.F(4), FxC03.F(-1), FxC03.F(-1), FxC03.F(-1) };

// (3/2)*diag(4,-1,-1,-1) = eq.79, coefficient par coefficient.
bool egalite79 = true;
for (int i = 0; i < 4; i++)
{
    var gauche = FxC03.F(3, 2) * G81[i];
    bool ok = gauche == G79[i];
    if (!ok) egalite79 = false;
    Console.WriteLine($"  (3/2)*G81[{i}] = {gauche} ; G79[{i}] = {G79[i]} -> {(ok ? "CONFORME" : "ECART")}");
    VerifC03.Egale(ok, $"eq.79 : (3/2)*G81[{i}] doit valoir {G79[i]}");
}
Console.WriteLine($"(3/2)*diag(4,-1,-1,-1) = eq.79 -> {(egalite79 ? "CONFORME" : "ECART")}");

// Les rapports unite/direction : -4 aux deux liens.
var ratio79 = G79[0] / G79[1];
var ratio81 = G81[0] / G81[1];
var moinsQuatre = new RationnelC03(-4);
Console.WriteLine($"ratio eq.79 : 6 / (-3/2) = {ratio79} ; ratio eq.81 : 4 / (-1) = {ratio81}");
VerifC03.Egale(ratio79 == moinsQuatre, $"ratio eq.79 : -4 attendu, obtenu {ratio79}");
VerifC03.Egale(ratio81 == moinsQuatre, $"ratio eq.81 : -4 attendu, obtenu {ratio81}");

// Signes des pivots et Sylvester sur les deux diagonales.
var sig79 = GramC12.SigPivots(G79);
var sig81 = GramC12.SigPivots(G81);
var syl79 = GramC12.SigSylvester(G79);
var syl81 = GramC12.SigSylvester(G81);
Console.WriteLine($"eq.79 : signes pivots ({sig79.Positifs}, {sig79.Negatifs}) ; Sylvester ({syl79.Positifs}, {syl79.Negatifs})");
Console.WriteLine($"eq.81 : signes pivots ({sig81.Positifs}, {sig81.Negatifs}) ; Sylvester ({syl81.Positifs}, {syl81.Negatifs})");
foreach (var cas in new[]
         {
             (nom: "eq.79 par pivots", calcule: sig79), (nom: "eq.81 par pivots", calcule: sig81),
             (nom: "eq.79 par Sylvester", calcule: syl79), (nom: "eq.81 par Sylvester", calcule: syl81),
         })
{
    bool ok = cas.calcule == (1, 3);
    Console.WriteLine($"  {cas.nom} : ({cas.calcule.Positifs}, {cas.calcule.Negatifs}) (attendu (1, 3)) -> {(ok ? "CONFORME" : "ECART")}");
    VerifC03.Egale(ok, $"{cas.nom} : signature (1, 3) attendue");
}
Console.WriteLine($"Grams mesurees : (3/2)*G81 = G79 -> {egalite79} ; rapport -4 aux deux liens -> {ratio79 == moinsQuatre && ratio81 == moinsQuatre} ; signature (1,3) par les deux methodes -> {sig79 == (1, 3) && sig81 == (1, 3) && syl79 == (1, 3) && syl81 == (1, 3)}");


  (3/2)*G81[0] = 6 ; G79[0] = 6 -> CONFORME


  (3/2)*G81[1] = -3/2 ; G79[1] = -3/2 -> CONFORME


  (3/2)*G81[2] = -3/2 ; G79[2] = -3/2 -> CONFORME


  (3/2)*G81[3] = -3/2 ; G79[3] = -3/2 -> CONFORME


(3/2)*diag(4,-1,-1,-1) = eq.79 -> CONFORME


ratio eq.79 : 6 / (-3/2) = -4 ; ratio eq.81 : 4 / (-1) = -4


... (3 outputs left out)

  eq.81 par pivots : (1, 3) (attendu (1, 3)) -> CONFORME


  eq.79 par Sylvester : (1, 3) (attendu (1, 3)) -> CONFORME


  eq.81 par Sylvester : (1, 3) (attendu (1, 3)) -> CONFORME


Grams mesurees : (3/2)*G81 = G79 -> True ; rapport -4 aux deux liens -> True ; signature (1,3) par les deux methodes -> True


### Lecture du résultat (Grams)

**Mesuré** : les deux Grams donnent la signature (1, 3) par les **deux méthodes des deux moteurs du source** — les signes des pivots (« the pivot signs at the first engine ») et les mineurs principaux (« the leading principal minors at the second », critère de Sylvester) — et le rapport $-4$ est commun aux deux liens. **Cité** : le calcul cellule par cellule, l'égalité à la rangée (2,2) aux deux bouts, 116/116 pas (§55).

## §55 — Théorème 55.1 : un contre trois à chaque pas de la chaîne

« At every chain and step carrying a (2,2) link, the Gram of the propagation reads one against three, the boost of the chain is its isometry with the mixing block nonzero, and the one-unit rotation [...] is the isometry of a definite plane and of no restriction of that Gram » (p. 146). **Standing: theorem.**

Le census (cité) : 34/36 états de chaîne parcourus à 116 pas sur 82 liens (2,2) série des neuf toiles enregistrées — l'électron, le muon, le tau, le proton, le neutron jusqu'au deutéron, le mot photon, la toile univers, le puits neutron et le puits tau. Nous, nous mesurons les **matrices imprimées** du plan : les deux générateurs, leurs deux plans, leurs carrés et leurs résidus exacts.

In [6]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Le plan (eq. 82 et figure 28) : l'unite hermitienne et une direction portee.
// le plan 4, 0 ; 0, -1 (indefini)
var PL = new[]
{
    new[] { FxC03.F(4), FxC03.F(0) },
    new[] { FxC03.F(0), FxC03.F(-1) },
};
// le plan des ordres 1, 0 ; 0, 1/4 (defini)
var Dp = new[]
{
    new[] { FxC03.F(1), FxC03.F(0) },
    new[] { FxC03.F(0), FxC03.F(1, 4) },
};
// le boost 0, 1/2 ; 2, 0
var Kp = new[]
{
    new[] { FxC03.F(0), FxC03.F(1, 2) },
    new[] { FxC03.F(2), FxC03.F(0) },
};
// la rotation d'unite 0, -1/2 ; 2, 0
var Rp = new[]
{
    new[] { FxC03.F(0), FxC03.F(-1, 2) },
    new[] { FxC03.F(2), FxC03.F(0) },
};
var I2 = new[]
{
    new[] { FxC03.F(1), FxC03.F(0) },
    new[] { FxC03.F(0), FxC03.F(1) },
};
var Z2 = new[]
{
    new[] { FxC03.F(0), FxC03.F(0) },
    new[] { FxC03.F(0), FxC03.F(0) },
};

// Residu lineaire d'algebre : A^T P + P A.
var rBoost = MatC03.Add(MatC03.Mul(MatC03.Transp(Kp), PL), MatC03.Mul(PL, Kp));
bool boostOk = MatC03.Eq(rBoost, Z2);
Console.WriteLine($"K^T PL + PL K = {MatC03.Aff(rBoost)} -> nul : {boostOk}");
VerifC03.Egale(boostOk, $"le residu du boost sur le plan doit etre nul, obtenu {MatC03.Aff(rBoost)}");

// Le carre du boost est l'identite.
bool kCarre = MatC03.Eq(MatC03.Mul(Kp, Kp), I2);
Console.WriteLine($"K^2 = {MatC03.Aff(MatC03.Mul(Kp, Kp))} -> I : {kCarre}");
VerifC03.Egale(kCarre, $"K^2 doit valoir I, obtenu {MatC03.Aff(MatC03.Mul(Kp, Kp))}");

// Le carre de la rotation d'unite est moins l'identite.
bool rCarre = MatC03.Eq(MatC03.Mul(Rp, Rp), MatC03.Neg(I2));
Console.WriteLine($"R^2 = {MatC03.Aff(MatC03.Mul(Rp, Rp))} -> -I : {rCarre}");
VerifC03.Egale(rCarre, $"R^2 doit valoir -I, obtenu {MatC03.Aff(MatC03.Mul(Rp, Rp))}");

// Sur le plan defini (appariement euclidien), le residu de la rotation est nul.
var rEucl = MatC03.Add(MatC03.Mul(MatC03.Transp(Rp), Dp), MatC03.Mul(Dp, Rp));
bool rEuclOk = MatC03.Eq(rEucl, Z2);
Console.WriteLine($"R^T D + D R = {MatC03.Aff(rEucl)} -> nul (euclidien) : {rEuclOk}");
VerifC03.Egale(rEuclOk, $"le residu euclidien de la rotation doit etre nul, obtenu {MatC03.Aff(rEucl)}");

// Sur la Gram restreinte, le residu de la rotation vaut [[0, -4], [-4, 0]] : pas une isometrie.
var attenduGram = new[]
{
    new[] { FxC03.F(0), FxC03.F(-4) },
    new[] { FxC03.F(-4), FxC03.F(0) },
};
var rGram = MatC03.Add(MatC03.Mul(MatC03.Transp(Rp), PL), MatC03.Mul(PL, Rp));
bool rGramOk = MatC03.Eq(rGram, attenduGram);
Console.WriteLine($"R^T PL + PL R = {MatC03.Aff(rGram)} -> attendu {MatC03.Aff(attenduGram)} : {rGramOk} ; non nul : pas une isometrie de la Gram");
VerifC03.Egale(rGramOk, $"le residu de la rotation sur la Gram doit valoir [[0, -4], [-4, 0]], obtenu {MatC03.Aff(rGram)}");
Console.WriteLine($"Figure 28 (eq. 82) : boost lorentzien {boostOk && kCarre} ; rotation euclidienne {rCarre && rEuclOk} ; restriction non isometrique {rGramOk}");


K^T PL + PL K = [0, 0; 0, 0] -> nul : True


K^2 = [1, 0; 0, 1] -> I : True


R^2 = [-1, 0; 0, -1] -> -I : True


R^T D + D R = [0, 0; 0, 0] -> nul (euclidien) : True


R^T PL + PL R = [0, -4; -4, 0] -> attendu [0, -4; -4, 0] : True ; non nul : pas une isometrie de la Gram


Figure 28 (eq. 82) : boost lorentzien True ; rotation euclidienne True ; restriction non isometrique True


### Lecture du résultat (le plan, ligne à ligne de la figure 28)

**Mesuré** : le boost $\begin{psmallmatrix} 0 & 1/2 \\ 2 & 0 \end{psmallmatrix}$ a un résidu nul sur $\mathrm{diag}(4,-1)$ et **carré l'identité** ; la rotation d'unité $\begin{psmallmatrix} 0 & -1/2 \\ 2 & 0 \end{psmallmatrix}$ a **carré moins l'identité**, résidu nul sur le plan défini $\mathrm{diag}(1, \tfrac14)$ (l'appariement euclidien), et sur la Gram restreinte son résidu est exactement $\begin{psmallmatrix} 0 & -4 \\ -4 & 0 \end{psmallmatrix}$ — non nul : **pas une isométrie**. La table de la figure 28 (p. 150), reproduite colonne par colonne : générateur, plan, carré, appariement préservé, résidu, lettre — Euclidien pour la rotation, Lorentzien pour le boost, aucune isométrie pour la restriction.

In [7]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Le releve au bout qubit : le triple litteral du ch. 55 (deja croise au ch. 49).
// Complexe exact : une paire de rationnels (partie reelle, partie imaginaire) -- jamais un double.
public readonly struct ComplexeC17 : IEquatable<ComplexeC17>
{
    public RationnelC03 Re { get; }
    public RationnelC03 Im { get; }

    public ComplexeC17(RationnelC03 re, RationnelC03 im) { Re = re; Im = im; }

    public static ComplexeC17 Zero => new ComplexeC17(new RationnelC03(BigInteger.Zero), new RationnelC03(BigInteger.Zero));

    public static ComplexeC17 operator +(ComplexeC17 a, ComplexeC17 b) =>
        new ComplexeC17(a.Re + b.Re, a.Im + b.Im);

    public static ComplexeC17 operator -(ComplexeC17 a, ComplexeC17 b) =>
        new ComplexeC17(a.Re - b.Re, a.Im - b.Im);

    public static ComplexeC17 operator *(ComplexeC17 a, ComplexeC17 b) =>
        new ComplexeC17(a.Re * b.Re - a.Im * b.Im, a.Re * b.Im + a.Im * b.Re);

    public static ComplexeC17 operator *(ComplexeC17 a, RationnelC03 k) =>
        new ComplexeC17(a.Re * k, a.Im * k);

    public static ComplexeC17 operator -(ComplexeC17 a) => new ComplexeC17(-a.Re, -a.Im);

    public ComplexeC17 Conjugue => new ComplexeC17(Re, -Im);

    public static bool operator ==(ComplexeC17 a, ComplexeC17 b) => a.Re == b.Re && a.Im == b.Im;

    public static bool operator !=(ComplexeC17 a, ComplexeC17 b) => !(a == b);

    public bool Equals(ComplexeC17 autre) => this == autre;

    public override bool Equals(object? obj) => obj is ComplexeC17 c && Equals(c);

    public override int GetHashCode() => HashCode.Combine(Re, Im);

    public override string ToString() => Im == new RationnelC03(BigInteger.Zero)
        ? Re.ToString()
        : "(" + Re.ToString() + ", " + Im.ToString() + "i)";
}

// Dagger et comparaison matricielle exacte sur des matrices 2x2 de complexes.
public static class ComplexesC17
{
    public static ComplexeC17[][] Dagger(ComplexeC17[][] M) =>
        new[]
        {
            new[] { M[0][0].Conjugue, M[1][0].Conjugue },
            new[] { M[0][1].Conjugue, M[1][1].Conjugue },
        };

    public static ComplexeC17[][] Neg(ComplexeC17[][] M) =>
        new[]
        {
            new[] { -M[0][0], -M[0][1] },
            new[] { -M[1][0], -M[1][1] },
        };

    public static bool Eq(ComplexeC17[][] A, ComplexeC17[][] B)
    {
        for (int i = 0; i < 2; i++)
            for (int j = 0; j < 2; j++)
                if (!(A[i][j] == B[i][j])) return false;
        return true;
    }

    public static string Aff(ComplexeC17[][] M) =>
        "[" + string.Join("; ", new[]
        {
            string.Join(", ", new[] { M[0][0].ToString(), M[0][1].ToString() }),
            string.Join(", ", new[] { M[1][0].ToString(), M[1][1].ToString() }),
        }) + "]";
}

// Le triple litteral du ch. 55 : C0 = 0, CI = i, et le scalonne d'un complexe.
var C0 = new ComplexeC17(new RationnelC03(BigInteger.Zero), new RationnelC03(BigInteger.Zero));
var CI = new ComplexeC17(new RationnelC03(BigInteger.Zero), new RationnelC03(BigInteger.One));
ComplexeC17 CScal(ComplexeC17 z, RationnelC03 n) => new ComplexeC17(z.Re * n, z.Im * n);

var moitie = FxC03.F(1, 2);
var moinsMoitie = FxC03.F(-1, 2);

var G1 = new[]
{
    new[] { C0, CScal(CI, moinsMoitie) },
    new[] { CScal(CI, moinsMoitie), C0 },
};
var G2 = new[]
{
    new[] { C0, new ComplexeC17(moitie, new RationnelC03(BigInteger.Zero)) },
    new[] { new ComplexeC17(moinsMoitie, new RationnelC03(BigInteger.Zero)), C0 },
};
var G3 = new[]
{
    new[] { CScal(CI, moinsMoitie), C0 },
    new[] { C0, CScal(CI, moitie) },
};
// re-orientation (flip G2)
var T = new[]
{
    G1,
    ComplexesC17.Neg(G2),
    G3,
};
var nomsTriple = new[] { "G1", "G2 (re-orientee)", "G3" };

Console.WriteLine("Le releve au bout qubit : le triple litteral du ch. 55 (donnee CITEE).");
int antiConformes = 0;
for (int k = 0; k < T.Length; k++)
{
    var d = ComplexesC17.Dagger(T[k]);
    var n = ComplexesC17.Neg(T[k]);
    bool ok = ComplexesC17.Eq(d, n);
    if (ok) antiConformes++;
    Console.WriteLine($"  {nomsTriple[k]} : dagger = {ComplexesC17.Aff(d)} ; -M = {ComplexesC17.Aff(n)} -> {(ok ? "CONFORME" : "ECART")}");
    VerifC03.Egale(ok, $"{nomsTriple[k]} doit etre anti-hermitienne (dagger = -M)");
}
bool anti = antiConformes == 3;
Console.WriteLine($"Les trois rotations anti-hermitiennes (dagger = -M) : {antiConformes} / 3 -> {(anti ? "CONFORME" : "ECART")}");
VerifC03.Egale(anti, "les trois rotations du triple litteral doivent etre anti-hermitiennes");
Console.WriteLine("Les trois boosts hermitiens au bout qubit : CITE -- le releve est celui de la");
Console.WriteLine("construction ; notre generateur de plan porte l'isometrie eta, mesuree plus haut");


Le releve au bout qubit : le triple litteral du ch. 55 (donnee CITEE).


  G1 : dagger = [0, (0, 1/2i); (0, 1/2i), 0] ; -M = [0, (0, 1/2i); (0, 1/2i), 0] -> CONFORME


  G2 (re-orientee) : dagger = [0, 1/2; -1/2, 0] ; -M = [0, 1/2; -1/2, 0] -> CONFORME


  G3 : dagger = [(0, 1/2i), 0; 0, (0, -1/2i)] ; -M = [(0, 1/2i), 0; 0, (0, -1/2i)] -> CONFORME


Les trois rotations anti-hermitiennes (dagger = -M) : 3 / 3 -> CONFORME


Les trois boosts hermitiens au bout qubit : CITE -- le releve est celui de la


construction ; notre generateur de plan porte l'isometrie eta, mesuree plus haut


### Lecture du résultat (le relevé)

**Mesuré** : les trois générateurs de rotation du triple littéral — celui-là même que la chaîne transporte (« the transposed triple », p. 147) — sont anti-hermitiens au dagger ordinaire : les « three anti-Hermitian » du théorème. **Cité** : les « three Hermitian generators » du côté boost — le relevé du bout qubit appartient à la construction, notre modèle porte l'isométrie $\eta$.

In [8]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Le cone nul du plan 4, 0 ; 0, -1, et le contraste Pell (rapport non carre).
// La forme q(t, x) = 4 t^2 - x^2 est evaluee en rationnel exact sur le plan PL.
var directions = new[]
{
    (t: new RationnelC03(BigInteger.One), x: new RationnelC03(BigInteger.One * 2)),
    (t: new RationnelC03(BigInteger.One), x: new RationnelC03(BigInteger.One * -2)),
};
var nulle = new RationnelC03(BigInteger.Zero);
int directionsNulles = 0;
foreach (var v in directions)
{
    var q = PL[0][0] * v.t * v.t + PL[1][1] * v.x * v.x;
    bool estNulle = q == nulle;
    if (estNulle) directionsNulles++;
    Console.WriteLine($"direction ({v.t}, {v.x}) : 4t^2 - x^2 = {q} -> {(estNulle ? "nulle" : "non nulle")}");
    VerifC03.Egale(estNulle, $"la direction ({v.t}, {v.x}) doit etre nulle sur le cone");
}
bool coneOk = directionsNulles == 2;
Console.WriteLine($"Cone nul : {directionsNulles} / 2 directions nulles -> {(coneOk ? "CONFORME" : "ECART")}");
VerifC03.Egale(coneOk, $"les deux directions fournies doivent etre nulles sur le cone, trouvees : {directionsNulles}");

// Le contraste de Pell : 3^2 - 2 * 2^2 = 1, la solution fondamentale de t^2 - 2 x^2 = 1.
var pell = new RationnelC03(BigInteger.One * 3) * new RationnelC03(BigInteger.One * 3)
            - new RationnelC03(BigInteger.One * 2) * new RationnelC03(BigInteger.One * 2) * new RationnelC03(BigInteger.One * 2);
var un = new RationnelC03(BigInteger.One);
Console.WriteLine($"Pell : 3^2 - 2*2^2 = {pell} -- la solution fondamentale de t^2 - 2x^2 = 1 (Chaines-09)");
VerifC03.Egale(pell == un, $"la solution de Pell attendue vaut 1, obtenu {pell}");
Console.WriteLine($"Contraste Pell : {pell} -> {(pell == un ? "CONFORME" : "ECART")}");

direction (1, 2) : 4t^2 - x^2 = 0 -> nulle


direction (1, -2) : 4t^2 - x^2 = 0 -> nulle


Cone nul : 2 / 2 directions nulles -> CONFORME


Pell : 3^2 - 2*2^2 = 1 -- la solution fondamentale de t^2 - 2x^2 = 1 (Chaines-09)


Contraste Pell : 1 -> CONFORME


### Lecture du résultat (le cône et Pell)

**Mesuré** : le plan de l'unité porte exactement deux directions nulles, $(1, 2)$ et $(1, -2)$ — le cône dessiné au-dessus de la chaîne (figure 27) ; et $3^2 - 2\cdot2^2 = 1$, la solution fondamentale de Pell. **Carrefour** : la famille de Pell donne une **infinité** de cartes au rapport non carré (Chaînes-09), quand le rapport carré donne un groupe **fini** — les 24 rotations du réseau de Chaînes-10. « while the Euclidean pairing admits a finite group at every ratio » (p. 145, cité) : nos deux carnets précédents en sont les deux moitiés.

## Ce que le census porte (§54, cité)

6 objets énumérés ; la Gram calculée cellule par cellule aux **six types de paire**, la scission (1, 3) partout ; le contre-test de l'unité frappée — l'algèbre de dimension six passe à l'algèbre de directions de dimension trois à la signature (0, 3), « with no plane of the unit » ; cinq contre-tests aux lettres épinglées ; treize contrôles lus pareil aux deux moteurs ; la reproduction par une seconde construction, égale à chaque cellule ; la plus longue chaîne de prémisses : douze chapitres.

In [9]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Numerics;

// Les comptes du census (§54), cites tels qu'imprimes (p. 145-146).
// Donnees citees : elles ne sont pas recalculees ici, elles sont saisies comme imprimees.
var census54 = new List<KeyValuePair<string, string>>
{
    new KeyValuePair<string, string>("objets enumeres", "6"),
    new KeyValuePair<string, string>("types de paire (Gram cellule par cellule)", "6"),
    new KeyValuePair<string, string>("contre-tests aux lettres epinglees", "5"),
    new KeyValuePair<string, string>("controles lus pareil aux deux moteurs", "13"),
    new KeyValuePair<string, string>("longueur de la plus longue chaine de premices (chapitres)", "12"),
};
foreach (var entree in census54)
    Console.WriteLine($"{entree.Key} : {entree.Value} (CITE)");
Console.WriteLine("contre-test nomme : unite frappee -> algebre de directions seule, dimension 3,");
Console.WriteLine("signature (0, 3), sans plan de l'unite (cite) ; notre moitie negative de kappa la lit");
Console.WriteLine($"Comptes cites du §54 saisis : {census54.Count} (CITE, non recalcules)");


objets enumeres : 6 (CITE)


types de paire (Gram cellule par cellule) : 6 (CITE)


contre-tests aux lettres epinglees : 5 (CITE)


controles lus pareil aux deux moteurs : 13 (CITE)


longueur de la plus longue chaine de premices (chapitres) : 12 (CITE)


contre-test nomme : unite frappee -> algebre de directions seule, dimension 3,


signature (0, 3), sans plan de l'unite (cite) ; notre moitie negative de kappa la lit


Comptes cites du §54 saisis : 5 (CITE, non recalcules)


## Ce que le census porte (§55, cité)

Un contre trois à **116/116** pas (défini à aucun, autre à aucun) ; le boost isométrie à 116/116 ; la rotation d'unité isométrie de son plan défini à 116/116 et **d'aucune restriction de la Gram** ; l'orientation de la triple portée **préservée aux pas pairs, renversée aux impairs** (50/116) — figure 26 : a1 p, x1 r, x2 p, x3 r, a2 p sur la chaîne de l'électron ; les constantes de structure **niées aux pas impairs** à la base littérale portée (78/90 au premier site inséré, les zéros tenant) et égales à la table des crochets à **90/90** à la base ré-orientée à chaque pas — c'est le flip de Chaînes-10 qui alterne avec la parité ; le pôle à chaque lien, fonction à deux points $(30+30\sqrt2)/2304$, 204/204 ; le résultat de boost ouvert transcrit intégralement et re-dérivé au cadre du lien (2,3), les deux comptes imprimés par la cellule de census qui suit.

Quatre contre-tests : l'unité frappée → le plan devient défini et le type rotation à chaque pas ; l'orientation renversée à chaque pas → les comptes bougent, la signature pas ; le lien (2,3) terminal du proton et du photon lu comme lien de chaîne → la Gram passe à $\mathrm{diag}(6,-1,-1,-1)$ à ces deux pas, signature inchangée ; la porte frappée → toutes les cellules absentes.

In [10]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;

// Les comptes du census (§55), cites tels qu'imprimes (p. 146-149).
// Donnees citees : elles ne sont pas recalculees ici, elles sont saisies comme imprimees.
var census55 = new List<KeyValuePair<string, string>>
{
    new KeyValuePair<string, string>("etats de chaine parcourus", "34 / 36"),
    new KeyValuePair<string, string>("pas marches", "116"),
    new KeyValuePair<string, string>("liens (2,2) serie", "82"),
    new KeyValuePair<string, string>("un contre trois", "116 / 116 (defini a aucun, autre a aucun)"),
    new KeyValuePair<string, string>("boost isometrie", "116 / 116"),
    new KeyValuePair<string, string>("rotation d'unite sur son plan defini", "116 / 116"),
    new KeyValuePair<string, string>("orientation preservee (pairs) / renversee (impairs)", "50 / 116"),
    new KeyValuePair<string, string>("constantes egales a la table (base re-orientee)", "90 / 90 a 116/116"),
    new KeyValuePair<string, string>("pole a deux points", "204 / 204"),
    new KeyValuePair<string, string>("resultat de boost ouvert transcrit / re-derive", "15 / 15 et 10 / 10"),
};
foreach (var entree in census55)
    Console.WriteLine($"{entree.Key} : {entree.Value} (CITE)");
var lettres = new[] { "a1 p", "x1 r", "x2 p", "x3 r", "a2 p" };
Console.WriteLine($"orientation de la chaine de l'electron (figure 26, citee) : {string.Join(", ", lettres)}");
Console.WriteLine($"Comptes cites du §55 saisis : {census55.Count} (CITE, non recalcules)");


etats de chaine parcourus : 34 / 36 (CITE)


pas marches : 116 (CITE)


liens (2,2) serie : 82 (CITE)


un contre trois : 116 / 116 (defini a aucun, autre a aucun) (CITE)


boost isometrie : 116 / 116 (CITE)


rotation d'unite sur son plan defini : 116 / 116 (CITE)


... (2 outputs left out)

pole a deux points : 204 / 204 (CITE)


resultat de boost ouvert transcrit / re-derive : 15 / 15 et 10 / 10 (CITE)


orientation de la chaine de l'electron (figure 26, citee) : a1 p, x1 r, x2 p, x3 r, a2 p


Comptes cites du §55 saisis : 10 (CITE, non recalcules)


## Ouvert — ligne à ligne

1. **L'unité de boost ouverte au ch. 56** : « the one unit open at chapter 56 » — le prochain carnet (Théorème 56.2, open result, et la famille de Pell à l'unité debout).
2. **Le résultat de boost ouvert** dont le 55.1 transcrit les cellules : ses prémisses (l'ouvert 92.1, les théorèmes 92.2 et 115.1) ne sont pas prouvées ici.
3. **Le pôle** : la fonction à deux points $(30+30\sqrt2)/2304$ citée telle quelle — non rejouée.
4. **La rangée d'insertion série** et **la rangée à deux points** : prémisses ouvertes citées du ch. 10.
5. **Les huit rapports carrés de tick** (§54) : l'élément d'ordre infini existe à chacun (Pell) ; le compte des cartes par rapport n'est pas rejoué ici.

### Exercice 1 — l'exponentielle du boost au point hyperbolique rationnel

$K^2 = I$ (mesuré ci-dessus) donne $\exp(\theta K) = \cosh\theta\, I + \sinh\theta\, K$. Au point pythagoricien $(3,4,5)$ : $\cosh = 5/4$, $\sinh = 3/4$. Construire $M$, vérifier la condition **de groupe** $M^T P_L M = P_L$ — quadratique, à distinguer du résidu linéaire d'algèbre — puis lire $\tanh = \sinh/\cosh$ et le rapport $x/t$ de la première colonne.

In [11]:
#nullable enable
using System;
using System.Globalization;
using System.Numerics;

// Exercice 1 : le point pythagoricien (3,4,5) donne cosh = 5/4 et sinh = 3/4.
var ch = FxC03.F(5, 4);
var sh = FxC03.F(3, 4);
Console.WriteLine($"cosh = {ch} ; sinh = {sh} (rationnels exacts)");
VerifC03.Egale(ch == FxC03.F(5, 4), $"cosh attendu : 5/4, obtenu {ch}");
VerifC03.Egale(sh == FxC03.F(3, 4), $"sinh attendu : 3/4, obtenu {sh}");

// TODO etudiant : ch*I2 + sh*Kp, puis mul(mul(transp(M), PL), M) == PL,
// puis tanh = sh/ch et le rapport M[1][0]/M[0][0]
RationnelC03[][]? M = null;
Console.WriteLine("Exercice a completer");


cosh = 5/4 ; sinh = 3/4 (rationnels exacts)


Exercice a completer


### Exercice 2 — les signatures des deux moitiés

La forme de Killing mesurée est diagonale par blocs. Extraire sa restriction au sous-espace des rotations $(J_1, J_2, J_3)$ et à celui des boosts $(K_1, K_2, K_3)$ : deux signatures attendues, (0, 3) et (3, 0) — l'algèbre de directions du contre-test, et son complément.

In [12]:
#nullable enable
using System;

// TODO etudiant
(int Positifs, int Negatifs)? sig_rot = null;
(int Positifs, int Negatifs)? sig_boost = null;
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 3 — le lien (2,3) lu comme lien de chaîne

Le troisième contre-test : lire le lien terminal (2,3) du proton et du photon comme un lien de chaîne déplace la Gram vers $\mathrm{diag}(6,-1,-1,-1)$ — l'entrée de l'unité passe à la dimension du lien. Calculer les signes pivots et Sylvester de cette Gram : la signature bouge-t-elle ? Et son rapport unité/direction ?

In [13]:
#nullable enable
using System;
using System.Globalization;
using System.Numerics;

// Exercice 3 : la Gram du lien (2,3) lu comme lien de chaine.
var G23 = new[] { FxC03.F(6), FxC03.F(-1), FxC03.F(-1), FxC03.F(-1) };
Console.WriteLine($"G23 = [{string.Join(", ", G23.Select(v => v.ToString()))}] (rationnels exacts)");
VerifC03.Egale(G23.Length == 4, $"G23 : 4 entrees attendues, obtenu {G23.Length}");
VerifC03.Egale(G23[0] == FxC03.F(6), $"G23[0] attendu : 6, obtenu {G23[0]}");
VerifC03.Egale(G23[1] == FxC03.F(-1), $"G23[1] attendu : -1, obtenu {G23[1]}");
VerifC03.Egale(G23[2] == FxC03.F(-1), $"G23[2] attendu : -1, obtenu {G23[2]}");
VerifC03.Egale(G23[3] == FxC03.F(-1), $"G23[3] attendu : -1, obtenu {G23[3]}");

// TODO etudiant : sig_pivots(G23) et sig_sylvester(G23), puis le ratio 6/(-1)
(int Positifs, int Negatifs)? sig_23 = null;
Console.WriteLine("Exercice a completer");


G23 = [6, -1, -1, -1] (rationnels exacts)


Exercice a completer


## Conclusion

Le groupe de Lorentz n'est pas plus posé que le groupe des rotations : les six générateurs tombent de la scission hermitienne, les constantes (80) sont exactes, la Killing sépare trois et trois, et le rapport $-4$ — l'unité contre quatre directions — tient aux deux types de lien. La signature un contre trois vit au choix de l'unité, et le transport la porte le long de chaque chaîne sans bouger (théorème 55.1) : la rotation d'unité reste euclidienne, le boost reste lorentzien, à chaque pas.

La suite (ch. 51) place le boost à l'unité : l'ouvert 56.2 et la famille de Pell à l'unité debout — grain 12.

In [14]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Releve final : les checks du carnet.
// Chaque verdict est recalcule a partir des mesures des cellules precedentes,
// puis verifie par une assertion qui leve sur ecart.
var checksFinal = new List<KeyValuePair<string, bool>>
{
    new KeyValuePair<string, bool>("six residus d'isometrie (eta) nuls", residus.Values.All(v => v)),
    new KeyValuePair<string, bool>("rot-rot [J1,J2]=J3 (x3)", rotRot.All(v => v)),
    new KeyValuePair<string, bool>("melange [Ji,Kj]=eps_ijk Kk (x9)", melangeOk),
    new KeyValuePair<string, bool>("boost-boost [K1,K2]=-J3 (x3)", boostBoost.All(v => v)),
    new KeyValuePair<string, bool>("Jacobi 216/216 triples", jacobi),
    new KeyValuePair<string, bool>("Killing diag(-4,-4,-4,4,4,4), sig (3,3)",
        kappaDiagOk && sigKappa == (3, 3)),
    new KeyValuePair<string, bool>("(3/2)*diag(4,..) = eq.79",
        Enumerable.Range(0, 4).All(i => FxC03.F(3, 2) * G81[i] == G79[i])),
    new KeyValuePair<string, bool>("ratio -4 aux deux liens",
        ratio79 == new RationnelC03(-4) && ratio81 == new RationnelC03(-4)),
    new KeyValuePair<string, bool>("signatures (1,3) par pivots ET Sylvester",
        sig79 == (1, 3) && sig81 == (1, 3) && syl79 == (1, 3) && syl81 == (1, 3)),
    new KeyValuePair<string, bool>("plan : residu boost nul, K^2=I, R^2=-I",
        boostOk && kCarre && rCarre),
    new KeyValuePair<string, bool>("R euclidien (residu nul) et non-Gram ([[0,-4],[-4,0]])",
        rEuclOk && rGramOk),
    new KeyValuePair<string, bool>("rotations anti-hermitiennes (x3)", anti),
    new KeyValuePair<string, bool>("cone nul (1,2) et (1,-2)", coneOk),
    new KeyValuePair<string, bool>("Pell 3^2 - 2*2^2 = 1", pell == new RationnelC03(1)),
};

int conformesFinal = 0;
foreach (var check in checksFinal)
{
    Console.WriteLine($"{check.Key} -> {(check.Value ? "CONFORME" : "ECART")}");
    VerifC03.Egale(check.Value, $"releve final : {check.Key}");
    if (check.Value) conformesFinal++;
}
Console.WriteLine($"Total : {conformesFinal} / {checksFinal.Count}");
VerifC03.Egale(conformesFinal == checksFinal.Count,
    $"releve final : {checksFinal.Count} checks attendus, {conformesFinal} conformes");
VerifC03.Egale(checksFinal.Count == 14,
    $"releve final : 14 checks attendus, obtenu {checksFinal.Count}");


six residus d'isometrie (eta) nuls -> CONFORME


rot-rot [J1,J2]=J3 (x3) -> CONFORME


melange [Ji,Kj]=eps_ijk Kk (x9) -> CONFORME


boost-boost [K1,K2]=-J3 (x3) -> CONFORME


Jacobi 216/216 triples -> CONFORME


Killing diag(-4,-4,-4,4,4,4), sig (3,3) -> CONFORME


... (5 outputs left out)

rotations anti-hermitiennes (x3) -> CONFORME


cone nul (1,2) et (1,-2) -> CONFORME


Pell 3^2 - 2*2^2 = 1 -> CONFORME


Total : 14 / 14


## À retenir

1. **Le groupe de Lorentz se dérive, comme les rotations** : la même scission hermitienne de la
   trace donne six générateurs — trois $J$ anti-hermitiens (les rotations), trois $K$ hermitiens
   (les boosts) — et les six résidus d'isométrie sont nuls.
2. **Les constantes de structure (80) sont exactes** : $[J_1, J_2] = J_3$, les neuf mélanges
   $[J_i, K_j] = \epsilon_{ijk} K_k$, $[K_1, K_2] = -J_3$ — et l'identité de Jacobi tient sur les
   $216$ triples de la base.
3. **La forme de Killing sépare trois et trois** : $\kappa = \mathrm{diag}(-4,-4,-4,\,4,4,4)$,
   signature $(3,3)$ par élimination symétrique exacte — les rotations portent $(0,3)$, les boosts
   $(3,0)$, et c'est la lecture du contre-test du source.
4. **Un contre trois aux deux liens** : $G_{79} = \tfrac32\,\mathrm{diag}(4,-1,-1,-1)$ et
   $G_{81} = \mathrm{diag}(4,-1,-1,-1)$, tous deux de signature $(1,3)$ par pivots **et** par
   Sylvester, avec le même rapport $-4$ — l'unité contre les quatre directions.
5. **La signature vit au choix de l'unité, et le transport la porte** : la rotation d'unité reste
   euclidienne ($R^2 = -I$, résidu nul, et sa « Gram » $[[0,-4],[-4,0]]$ n'en est pas une), le
   boost reste lorentzien ($K^2 = I$), le cône nul porte $(1,2)$ et $(1,-2)$, et
   $3^2 - 2 \cdot 2^2 = 1$ est la famille de Pell à l'unité debout.

## Vérifiez votre compréhension

1. Pourquoi les rotations sont-elles anti-hermitiennes et les boosts hermitiens, et que mesure ce
   partage ?
2. Que vaut la forme de Killing sur cette algèbre, et comment s'y lit la signature $(3,3)$ ?
3. Les deux Grams de l'éq. 79 et de l'éq. 81 n'ont pas les mêmes coefficients : pourquoi donnent-ils
   pourtant la même signature ?
4. Qu'est-ce qui distingue le plan de la rotation d'unité de celui du boost, et où le théorème 55.1
   intervient-il ?

<details>
<summary>Réponses</summary>

1. Elles tombent de la scission hermitienne : $J = i\sigma/2$ est anti-hermitien, $K = \sigma/2$
   est hermitien. Le partage est exactement ce que la forme de Killing mesure — $\kappa(J,J) = -4$,
   $\kappa(K,K) = +4$, tous les croisés nuls.
2. $\kappa = \mathrm{diag}(-4,-4,-4,\,4,4,4)$ : trois négatifs (les rotations) et trois positifs
   (les boosts), d'où la signature $(3,3)$ — et, séparément, $(0,3)$ sur les rotations et $(3,0)$
   sur les boosts.
3. Ils ne diffèrent que par le facteur scalaire $3/2$ : $G_{79} = \tfrac32 G_{81}$. Un facteur
   positif ne change ni le signe ni le nombre des pivots, donc les deux formes ont la même
   signature $(1,3)$ — et le rapport des deux vaut $-4$ dans les deux cas.
4. La rotation d'unité vérifie $R^2 = -I$ — elle est euclidienne, de résidu nul, et sa forme
   associée $[[0,-4],[-4,0]]$ n'est pas une Gram (non définie) ; le boost vérifie $K^2 = I$ et son
   cône nul porte exactement $(1,2)$ et $(1,-2)$. Le théorème 55.1 est le transport : la signature
   un-contre-trois ne bouge pas le long de la chaîne.

</details>

## Pour aller plus loin

*Chaînes-12 — les boosts et l'unité une* est le
carnet que la conclusion annonce : le boost à l'unité, l'ouvert 56.2 et la famille de Pell à l'unité
debout.

En amont, [Chaînes-10 — le groupe des rotations](Chaines-10-Groupe-Rotations-Automorphismes-Directions-Python.ipynb)
est le carnet dont la conclusion annonçait celui-ci (« the next chapter joins them to the order
coordinate in the Lorentz group »), et [Chaînes-09 — la coordonnée d'ordre](Chaines-09-Coordonnee-Ordre-Chaine-Serie-Python.ipynb)
fournit la coordonnée que le théorème 55.1 transporte le long de la chaîne.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
reste le contraste de méthode : là où une conservation se dérive d'une symétrie continue, les six
générateurs de ce carnet sont **dérivés d'une scission**, et c'est la forme de Killing — non un
postulat — qui sépare les trois rotations des trois boosts.

**Version Python** de ce carnet : [Chaînes-11 — le groupe de Lorentz et la signature un contre trois (Python)](Chaines-11-Groupe-Lorentz-Signature-Un-Contre-Trois-Python.ipynb) — même fil, mêmes entrées, avec `fractions.Fraction` ; le jumeau C# utilise `System.Numerics.BigInteger` et un rationnel exact réduit.